In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")


# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_csv("military_raw_data.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())


# ==========================================================
# DATASET INFORMATION
# ==========================================================

print("=" * 60)
print("MODULE 2 - DATASET INFORMATION")
print("=" * 60)

print("\nShape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData Types Summary:\n")
print(df.dtypes.value_counts())


# ==========================================================
# MISSING VALUES REPORT
# ==========================================================

missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum()/len(df)*100).round(2)
})

missing = missing.sort_values(
    "Missing Values",
    ascending=False
)

display(missing[missing["Missing Values"] > 0])


# ==========================================================
# DUPLICATE CHECK
# ==========================================================

duplicates = df.duplicated(subset=["Country Name"]).sum()

print("Duplicate Countries:", duplicates)


# ==========================================================
# OBJECT COLUMNS
# ==========================================================

object_columns = df.select_dtypes(include="object").columns.tolist()

print("Total Object Columns:", len(object_columns))

for col in object_columns:
    print(col)


# ==========================================================
# RELOAD ORIGINAL DATA
# ==========================================================

df = pd.read_csv("military_raw_data.csv")

print("Dataset reloaded successfully!")
print(df.shape)



# ==========================================================
# NUMERIC CLEANING
# ==========================================================

text_columns = [
    "Country Name",
    "continent",
    "region",
    "alliance"
]


def clean_numeric_value(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    value = re.sub(r"[^0-9.\-]", "", value)

    if value == "":
        return np.nan

    return value



for col in df.columns:

    if col not in text_columns:
        df[col] = df[col].apply(clean_numeric_value)



print("Universal cleaning completed!")



# ==========================================================
# CONVERT NUMERIC TYPES
# ==========================================================

for col in df.columns:

    if col not in text_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


print("Numeric conversion completed!")



# ==========================================================
# NAVAL TONNAGE PREDICTION
# ==========================================================

target = "total_naval_fleet_tonnage_mt"


features = [
    "aircraft_carriers",
    "helicopter_carriers",
    "submarines",
    "destroyers",
    "frigates",
    "corvettes",
    "coastal_patrol_craft",
    "mine_warfare_craft",
    "total_naval_fleet"
]


train_df = df[df[target].notna()].copy()

predict_df = df[df[target].isna()].copy()


print("Training Countries:", len(train_df))
print("Prediction Countries:", len(predict_df))


model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)


X_train = train_df[features]
y_train = train_df[target]


model.fit(
    X_train,
    y_train
)


print("Naval model trained successfully!")



X_predict = predict_df[features]


predictions = model.predict(X_predict)


df.loc[
    df[target].isna(),
    target
] = predictions


print("Missing naval tonnage filled!")



# ==========================================================
# COMBINED PERSONNEL
# ==========================================================

df["combined_personnel"] = (
    df["active_personnel"] +
    df["reserve_personnel"] +
    df["paramilitary"]
)


print("Combined personnel created!")



# ==========================================================
# COALITION BUDGET
# ==========================================================

nato_total_budget = df.loc[
    df["alliance"]=="NATO",
    "defense_budget_usd"
].sum()


df["coalition_budget"] = df["defense_budget_usd"]


df.loc[
    df["alliance"]=="NATO",
    "coalition_budget"
] = (
    df.loc[df["alliance"]=="NATO","defense_budget_usd"]
    +
    (0.05*nato_total_budget)
)


print("Coalition budget calculated!")



# ==========================================================
# FORCE READINESS INDEX
# ==========================================================

scaler = MinMaxScaler()


readiness_features = [
    "active_personnel",
    "combined_personnel",
    "total_military_manpower"
]


scaled = scaler.fit_transform(
    df[readiness_features]
)


scaled_df = pd.DataFrame(
    scaled,
    columns=readiness_features
)


df["force_readiness_index"] = (
    scaled_df["active_personnel"]*0.40 +
    scaled_df["combined_personnel"]*0.30 +
    scaled_df["total_military_manpower"]*0.30
)*100



# ==========================================================
# AIR SUPERIORITY INDEX
# ==========================================================

air_features = [
    "fighter_aircraft",
    "attack_aircraft",
    "total_military_aircraft"
]


scaled = scaler.fit_transform(
    df[air_features]
)


scaled_df = pd.DataFrame(
    scaled,
    columns=air_features
)


df["air_superiority_index"] = (
    scaled_df["fighter_aircraft"]*0.45 +
    scaled_df["attack_aircraft"]*0.25 +
    scaled_df["total_military_aircraft"]*0.30
)*100



# ==========================================================
# LOGISTICS INDEX
# ==========================================================

logistics_features = [
    "total_serviceable_airports",
    "total_merchant_marine_fleet",
    "railway_coverage_km",
    "roadway_coverage_km",
    "major_ports_and_terminals"
]


scaled = scaler.fit_transform(
    df[logistics_features]
)


scaled_df = pd.DataFrame(
    scaled,
    columns=logistics_features
)



df["logistics_index"] = (
    scaled_df["total_serviceable_airports"]*0.25 +
    scaled_df["total_merchant_marine_fleet"]*0.20 +
    scaled_df["railway_coverage_km"]*0.20 +
    scaled_df["roadway_coverage_km"]*0.20 +
    scaled_df["major_ports_and_terminals"]*0.15
)*100



# ==========================================================
# SUPERIORITY INDEX
# ==========================================================

df["superiority_index"] = (
    df["force_readiness_index"]*0.40 +
    df["air_superiority_index"]*0.30 +
    df["logistics_index"]*0.30
)



# ==========================================================
# BUDGET ADVANTAGE
# ==========================================================

budget_ratio = (
    df["defense_budget_usd"] /
    df["gdp_usd"]
).fillna(0)


df["budget_advantage"] = (
    MinMaxScaler()
    .fit_transform(
        budget_ratio.values.reshape(-1,1)
    )
    *100
)



# ==========================================================
# POWER ADVANTAGE
# ==========================================================

power_ratio = (
    df["combined_personnel"] /
    df["total_population"]
).fillna(0)


df["power_advantage"] = (
    MinMaxScaler()
    .fit_transform(
        power_ratio.values.reshape(-1,1)
    )
    *100
)



# ==========================================================
# COALITION SCORE
# ==========================================================

coalition_budget_score = (
    MinMaxScaler()
    .fit_transform(
        df[["coalition_budget"]]
    )
)


df["coalition_score"] = (
    df["superiority_index"]*0.60 +
    coalition_budget_score.flatten()*100*0.40
)



# ==========================================================
# POWER INDEX RANK GAP
# ==========================================================

df["power_index_rank_gap"] = (
    df["superiority_index"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
    -1
)



# ==========================================================
# VALIDATION
# ==========================================================

print("="*60)
print("MODULE 2 VALIDATION")
print("="*60)


derived_columns = [
    "combined_personnel",
    "coalition_budget",
    "force_readiness_index",
    "air_superiority_index",
    "logistics_index",
    "superiority_index",
    "budget_advantage",
    "power_advantage",
    "coalition_score",
    "power_index_rank_gap"
]


print(df[derived_columns].isna().sum())


print("\nRemaining Missing Values:")
print(df.isna().sum()[df.isna().sum()>0])


print("\nDataset Shape:", df.shape)



# ==========================================================
# EXPORT DATASET
# ==========================================================

df.to_csv(
    "military_cleaned.csv",
    index=False
)


print("military_cleaned.csv created successfully!")

Libraries imported successfully!
Dataset loaded successfully!
Shape: (145, 69)


,Country Name,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,fighter_aircraft,attack_aircraft,transport_aircraft,trainer_aircraft,special_mission_aircraft,tanker_aircraft,total_military_helicopters,attack_helicopters,tanks,armored_fighting_vehicles,self_propelled_artillery,towed_artillery,rocket_projectors,total_naval_fleet,total_naval_fleet_tonnage_mt,aircraft_carriers,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,mine_warfare_craft,defense_budget_usd,external_debt_usd,purchasing_power_parity_usd,foreign_exchange_and_gold_reserves_usd,total_serviceable_airports,labour_force,major_ports_and_terminals,total_merchant_marine_fleet,railway_coverage_km,roadway_coverage_km,oil_production_bbl,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km,continent,region,gdp_usd,alliance,power_index_rank_gap,force_readiness_index,air_superiority_index,logistics_index,superiority_index,coalition_score,coalition_budget,combined_personnel,power_advantage,budget_advantage
0,Afghanistan,40121552,15647405,8826741,842553,75000,0,90000,5,0,0,2,0,0,0,3,0,0,3902,0,0,0,0,NaN,0,0,0,0,0,0,0,0,$ \t\t\t\t\t\t\t145000000,$ \t\t\t\t\t\t\t1929750000,$ \t\t\t\t\t\t\t8223800...,$ \t\t\t\t\t\t\t8852092000,68,9133000,0,0,0 \t\t\t\t\t\t\r\n\t\t\...,34903 \t\t\t\t\t\t\r\n\...,0 \t\t\t\t\t\t\r\n\t\t\...,58000 \t\t\t\t\t\t\r\n\...,0 \t\t\t\t\t\t\r\n\t\t\...,80200000 \t\t\t\t\t\t\r...,80200000 \t\t\t\t\t\t\r...,49554000000 \t\t\t\t\t\...,767000 \t\t\t\t\t\t\r\n...,503000 \t\t\t\t\t\t\r\n...,66000000 \t\t\t\t\t\t\r...,652230 \t\t\t\t\t\t\r\n...,0 \t\t\t\t\t\t\r\n\t\t\...,5987 \t\t\t\t\t\t\r\n\t...,1200 \t\t\t\t\t\t\r\n\t...,Asia,Southern Asia,1.458314e+10,Non-NATO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,3107100,1522479,1292554,62142,7500,0,500,20,0,0,1,0,0,0,19,0,0,1492,10,0,270,33,NaN,0,0,0,0,0,0,33,0,$ \t\t\t\t\t\t\t720037190,$ \t\t\t\t\t\t\t5363000000,$ \t\t\t\t\t\t\t5136000...,$ \t\t\t\t\t\t\t6516000000,3,1370000,3,69,424 \t\t\t\t\t\t\r\n\t\...,3581 \t\t\t\t\t\t\r\n\t...,14000 \t\t\t\t\t\t\r\n\...,21000 \t\t\t\t\t\t\r\n\...,150000000 \t\t\t\t\t\t\...,50623000 \t\t\t\t\t\t\r...,50623000 \t\t\t\t\t\t\r...,5692000000 \t\t\t\t\t\t...,473000 \t\t\t\t\t\t\r\n...,255000 \t\t\t\t\t\t\r\n...,522000000 \t\t\t\t\t\t\...,28748 \t\t\t\t\t\t\r\n\...,362 \t\t\t\t\t\t\r\n\t\...,691 \t\t\t\t\t\t\r\n\t\...,41 \t\t\t\t\t\t\r\n\t\t...,Europe,Southern Europe,1.888210e+10,NATO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,47022473,22570787,19185169,752360,130000,150000,150000,620,111,42,63,90,10,4,300,74,1485,24920,224,483,188,111,83480.0,0,0,6,0,8,8,75,3,$ \t\t\t\t\t\t\t2500000...,$ \t\t\t\t\t\t\t4764000000,$ \t\t\t\t\t\t\t7229120...,$ \t\t\t\t\t\t\t8300700...,95,13294000,17,119,4020 \t\t\t\t\t\t\r\n\t...,112696 \t\t\t\t\t\t\r\n...,1443000 \t\t\t\t\t\t\r\...,446000 \t\t\t\t\t\t\r\n...,12200000000 \t\t\t\t\t\...,100726000000 \t\t\t\t\t...,47963000000 \t\t\t\t\t\...,4504000000000 \t\t\t\t\...,0 \t\t\t\t\t\t\r\n\t\t\...,3000 \t\t\t\t\t\t\r\n\t...,233000000 \t\t\t\t\t\t\...,2381740 \t\t\t\t\t\t\r\...,998 \t\t\t\t\t\t\r\n\t\...,6734 \t\t\t\t\t\t\r\n\t...,0 \t\t\t\t\t\t\r\n\t\t\...,Africa,Northern Africa,1.920000e+11,Non-NATO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,37202061,7440412,3720206,372021,107000,0,10000,278,67,6,32,56,2,0,115,14,319,2258,25,575,116,32,NaN,0,0,0,0,0,0,32,0,$ \t\t\t\t\t\t\t3120000...,$ \t\t\t\t\t\t\t4529900...,$ \t\t\t\t\t\t\t2782390...,$ \t\t\t\t\t\t\t1424300...,107,15961000,21,64,2761 \t\t\t\t\t\t\r\n\t...,76000 \t\t\t\t\t\t\r\n\...,1175000 \t\t\t\t\t\t\r\...,121000 \t\t\t\t\t\t\r\n...,7783000000 \t\t\t\t\t\t...,5514000000 \t\t\t\t\t\t...,1397000000 \t\t\t\t\t\t...,3

MODULE 2 - DATASET INFORMATION

Shape: (145, 69)
Rows: 145
Columns: 69

Data Types Summary:

int64      34
object     23
float64    12
Name: count, dtype: int64


,Missing Values,Missing Percentage
coalition_score,145,100.00
coalition_budget,145,100.00
combined_personnel,145,100.00
budget_advantage,145,100.00
power_advantage,145,100.00
logistics_index,145,100.00
air_superiority_index,145,100.00
power_index_rank_gap,145,100.00
force_readiness_index,145,100.00
superiority_index,145,100.00


Duplicate Countries: 0
Total Object Columns: 23
Country Name
defense_budget_usd
external_debt_usd
purchasing_power_parity_usd
foreign_exchange_and_gold_reserves_usd
railway_coverage_km
roadway_coverage_km
oil_production_bbl
oil_consumption_bbl
proven_oil_reserves_bbl
natural_gas_production_cum
natural_gas_consumption_cum
proven_natural_gas_reserves_cum
coal_production_cum
coal_consumption_mt
proven_coal_reserves_cum
total_land_area_sq_km
coastline_coverage_km
border_coverage_km
waterway_coverage_km
continent
region
alliance
Dataset reloaded successfully!
(145, 69)
Universal cleaning completed!
Numeric conversion completed!
Training Countries: 51
Prediction Countries: 94
Naval model trained successfully!
Missing naval tonnage filled!
Combined personnel created!
Coalition budget calculated!
MODULE 2 VALIDATION
combined_personnel       0
coalition_budget         0
force_readiness_index    0
air_superiority_index    0
logistics_index          0
superiority_index        0
budget_advantage  

/tmp/ipykernel_407/3829570472.py:236: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[7.28236362e+10 8.70035990e+10 7.45889490e+10 1.17928559e+11
 7.48475990e+10 8.63535990e+10 7.44785990e+10 8.09935990e+10
 1.39335994e+11 1.99503689e+11 8.03457490e+10 7.79908490e+10
 7.26035990e+10 1.09428764e+11 7.21455990e+10 7.77553590e+10
 7.36342840e+10 7.23075990e+10 1.03894749e+11 7.25251261e+10
 8.32930822e+10 1.27968741e+11 7.88635990e+10 8.23342290e+10
 7.50035990e+10 7.38855990e+10 1.11104275e+11 8.80035990e+10
 1.60630129e+11 9.03603599e+11]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[
